# MiniGPT: a GPT built from scratch

Runs the whole project top to bottom: tests, the attention hand calculation, training, evaluation, attention inspection, ablations and an API call.

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then run the cells in order. Cells marked *(optional, slow)* can be skipped.

## 0. Setup

In [ ]:
import os
if not os.path.exists('llm-from-scratch'):
    !git clone https://github.com/syedukkashah/llm-from-scratch
%cd llm-from-scratch
!pip install -q -r requirements-dev.txt

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. Unit tests
Attention numerics, causal mask, shapes, tokenizer, training overfit, API.

In [ ]:
!python -m pytest -q

## 2. Attention by hand (Milestone 1)
A 3-token example with d_k = 2, computed step by step in plain Python, then compared with the PyTorch code.

In [ ]:
!python -m experiments.hand_calculation
from IPython.display import Image, display
display(Image('outputs/figures/fig1_2_hand_attention.png'))

In [ ]:
!python -m experiments.why_transformers
from IPython.display import Image, display
display(Image('outputs/figures/why_transformers.png'))

## 3. Multi-head attention and positions (Milestone 2)

In [ ]:
!python -m experiments.shape_trace

In [ ]:
!python -m experiments.positional_viz
from IPython.display import Image, display
display(Image('outputs/figures/fig4_positional_encoding.png'))

## 4. Transformer block and BERT vs GPT masks (Milestone 3)

In [ ]:
!python -m experiments.block_diagram
from IPython.display import Image, display
display(Image('outputs/figures/transformer_block.png'))

In [ ]:
!python -m experiments.mask_comparison
from IPython.display import Image, display
display(Image('outputs/figures/fig2_mask_comparison.png'))
from IPython.display import Image, display
display(Image('outputs/figures/mask_cheating_loss.png'))

## 5. Debugging gate: overfit one tiny batch (Milestone 4)
Loss must fall close to zero.

In [ ]:
!python -m src.train --overfit --d_model 64 --n_layers 2 --ffn_dim 256 --context_length 32 --lr 0.003

## 6. Train the model (Milestone 5)
Set `TRAIN = True` to train from scratch (about 7 minutes on a T4). With `False` the released checkpoint is downloaded automatically in the next cells.

In [ ]:
TRAIN = False
if TRAIN:
    !python -m src.train

## 7. Evaluate and generate

In [ ]:
!python -m experiments.evaluate
from IPython.display import Image, display
display(Image('outputs/figures/fig6_loss_curves.png'))

In [ ]:
!python -m src.generate --prompt "ROMEO:" --max-new-tokens 80 --seed 1
!python -m src.generate --prompt "To be, or not to be" --max-new-tokens 80 --seed 1
!python -m src.generate --prompt "KING HENRY:" --max-new-tokens 80 --seed 1

## 8. Inside the trained model: attention maps

In [ ]:
!python -m experiments.attention_visualization
from IPython.display import Image, display
display(Image('outputs/figures/fig7_attention.png'))

## 9. Ablations (Milestone 6)
Each experiment changes one factor (layers, heads, context length, position encoding) with the same seed, data and 2000 steps. The baseline is trained once and shared. This trains 8 models (a few minutes each on a T4), then compares sampling settings (greedy vs temperature, no training). *(optional, slow)*

To run only one experiment, replace `all` with `depth`, `heads`, `context`, `position` or `sampling`.

In [ ]:
!python -m experiments.ablation --experiment all

In [ ]:
print(open('outputs/logs/ablation_table.md').read())
from IPython.display import Image, display
display(Image('outputs/figures/ablation_depth.png'))
from IPython.display import Image, display
display(Image('outputs/figures/ablation_heads.png'))
from IPython.display import Image, display
display(Image('outputs/figures/ablation_context.png'))
from IPython.display import Image, display
display(Image('outputs/figures/ablation_position.png'))
from IPython.display import Image, display
display(Image('outputs/figures/ablation_sampling.png'))

## 10. The REST API
Starts the FastAPI server inside this notebook and calls it.

In [ ]:
import subprocess, time
server = subprocess.Popen(['python', '-m', 'uvicorn', 'src.api:app', '--port', '8000'])
time.sleep(10)
!curl -s localhost:8000/health
print()
!curl -s -X POST localhost:8000/generate -H 'Content-Type: application/json' -d '{"prompt": "ROMEO:", "max_new_tokens": 80, "seed": 1}'
server.terminate()

## 11. Download your results (optional)

In [ ]:
!zip -r results.zip checkpoints/*.pt outputs
from google.colab import files
files.download('results.zip')